$$
\boldsymbol{x}\in\mathbb{R}^{7168}
\xrightarrow{\mathbf W^{\downarrow}}
\boldsymbol{z}\in\mathbb{R}^{3584}
\xrightarrow{\mathbf W_g,\mathbf W_u}
(\boldsymbol a,\boldsymbol b)\in\mathbb{R}^{3072}\times\mathbb{R}^{3072}
\xrightarrow{\text{GLU}}
\boldsymbol f\in\mathbb{R}^{3072}
\xrightarrow{\mathbf W_2}
\boldsymbol e\in\mathbb{R}^{3584}
\xrightarrow{\text{Top-16 聚合、RMSNorm、}\mathbf W^{\uparrow}}
\boldsymbol y\in\mathbb{R}^{7168}
$$

- 也就是沿一条有效数据路径经历：$\mathbf W^{\downarrow}\rightarrow(\mathbf W_g/\mathbf W_u)\rightarrow\mathbf W_2\rightarrow\mathbf W^{\uparrow}$
    - “接近四次连续矩阵乘法”
        - 四个矩阵中间近乎线性串联，谱范数逐级相乘，放大效应随深度复利累积，93 层、2.8T 参数下路由分支的内部激活出现爆炸。
    - 这种条件较差的长投影链，加上 2.8T 参数规模，会在 routed branch 中产生内部激活爆炸（exploding internal activations）。
- $\boldsymbol a=\mathbf W_g\boldsymbol x,\qquad \boldsymbol b=\mathbf W_u\boldsymbol x$
- 标准 SwiGLU 为：$\operatorname{SwiGLU}(\boldsymbol a,\boldsymbol b)=[\boldsymbol a\odot\sigma(\boldsymbol a)]\odot\boldsymbol b$
- 问题是两个乘法因子都无界：
    - gate 分支 $\boldsymbol a\sigma(\boldsymbol a)$ 在 $a\to+\infty$ 时约等于 $a$；
    - up 分支 $b$ 是线性的，同样无界。
    - 同值切片 $a=b=t$ 上：
        - $f_{\text{SwiGLU}}(t)=t^2\sigma(t)\underset{t\to+\infty}{\sim}t^2$

### qb

- aux-loss（要在模型质量与均衡之间权衡）→ DeepSeek V3 的 bias update（$b_j \mathrel{+}= \gamma,\mathrm{sign}(\bar\ell-\ell_j)$，只知过冷过热，$\gamma$ 仍是超参）→ QB。
- 每个 token 加上当前 bias 后排在第 17 名的分数就是进 Top-16 的门槛 $\alpha_i$（$k=16$，Top-$(k{+}1)$）；对专家 $j$ 看它在整个 batch 里距各门槛的 margin $s_{i,j}-\alpha_i$，取分位数直接算出新 bias，使约 $16/896$ 的 token 会选中它；

- 8 个 token、4 个路由专家、每 token 选 1 个（$m=8,;n=4,;k=1$），目标负载 $q = mk/n = 2$。
    - 对应到 K3：$n=896,;k=16$，门槛取自 Top-17 而不是 Top-2，分位数 $1-k/n = 1-16/896 = 0.98214$，目标负载 $q = m/56$。机制完全相同，只是 $k$ 从 1 换成 16。
- 输入：路由分数 $\boldsymbol s = \mathrm{Sigmoid}(\mathbf W_r\boldsymbol x)\in(0,1)$，以及第 $t$ 步的偏置 $\boldsymbol b^{(t)} = (-0.08,;0,;+0.08,;0)$：
    - $s_{i,j}$: $i$, token index; $j$: expert index
    - $\alpha_i$: token wise, $b_j$: expert wise